# 05 Feature Engineering

In CRISP-DM Phase 3b werden auf Basis der NB04-Diagnostik neue Features konstruiert,
die das Modell in NB07 nutzen kann. Ziel ist eine Anreicherung des Feature-Sets mit
klinisch begründeten Kombinationsmerkmalen, Interaktionstermen und nichtlinearen
Transformationen — ohne Datenleck in das Test-Set.

**Architektur-Prinzip:**
Das Testset wird in diesem Notebook nicht transformiert und nicht angefasst.
Alle Transformationen werden ausschließlich auf `X_train` definiert und gefittet.
Die Anwendung auf `X_test` erfolgt später innerhalb der Modellierungs-Pipelines in NB07.
`X_test` wird hier weiterhin geladen (damit der Split dokumentiert ist), aber nicht
verändert und nicht exportiert — nur `X_train_enriched.parquet` wird persistiert.

**Weitere Regeln:**
- Kein Resampling in diesem Notebook.
- Transformer werden ausschließlich auf `X_train` gefittet.

**Struktur:**
1. Setup
2. Daten laden
3. Transformationen auf X_train anwenden
4. Composite Features
5. Interaktionsterme
6. Polynomterme und nichtlineare Transformationen
7. IV/MI Vorher-Nachher-Vergleich
8. Export
9. Zusammenfassung

## 1. Setup

Alle Bibliotheken werden zentral importiert. `optbinning` wird für den IV-Vergleich
in Abschnitt 7 benötigt. Der globale Seed `SEED = 42` wird konsistent aus NB03 übernommen.

In [ ]:
import subprocess, sys, importlib

for pkg in ['optbinning']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import PowerTransformer
import joblib
from optbinning import OptimalBinning

sys.path.insert(0, str(Path('..').resolve()))
from src.utils import cap_bmi, categorize_bmi, hurdle_encode

SEED = 42
np.random.seed(SEED)

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models/transformers')
OUTPUTS_DIR   = Path('../outputs/05_feature_engineering')
for d in [MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Setup abgeschlossen.')
print(f'  PROCESSED_DIR : {PROCESSED_DIR.resolve()}')
print(f'  MODELS_DIR    : {MODELS_DIR.resolve()}')
print(f'  OUTPUTS_DIR   : {OUTPUTS_DIR.resolve()}')

## 2. Daten laden

Die in NB03 erzeugten Parquet-Dateien werden direkt geladen. Der Split ist bereits
fixiert — kein neuerlicher `train_test_split`. `X_test` und `y_test` werden geladen,
damit der Split nachvollziehbar dokumentiert ist, aber in diesem Notebook weder
transformiert noch exportiert.

In [ ]:
X_train = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
X_test  = pd.read_parquet(PROCESSED_DIR / 'X_test.parquet')
y_train = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(PROCESSED_DIR / 'y_test.parquet').squeeze()

with open(PROCESSED_DIR / 'feature_meta.json') as f:
    meta = json.load(f)

BINARY_COLS  = meta['binary_cols']
ORDINAL_COLS = meta['ordinal_cols']
COUNT_COLS   = meta['count_cols']
NUMERIC_COLS = meta['numeric_cols']

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train Prävalenz: {y_train.mean():.4f}')
print(f'y_test  Prävalenz: {y_test.mean():.4f}')
print(f'\nFeature-Gruppen:')
print(f'  BINARY  : {len(BINARY_COLS)}')
print(f'  ORDINAL : {len(ORDINAL_COLS)}')
print(f'  COUNT   : {len(COUNT_COLS)}')
print(f'  NUMERIC : {len(NUMERIC_COLS)}')

In [ ]:
# Hilfsfunktion: IV und MI für ein einzelnes Feature berechnen
_iv_nan_log = []  # sammelt Features, bei denen OptimalBinning fehlschlug

def compute_iv(feat_series, y_series, feature_name, is_continuous=False):
    """IV via OptimalBinning (cp → mip Fallback). NaN wird geloggt."""
    X_arr = feat_series.values.astype(float)
    y_arr = y_series.values.astype(int)
    for solver in ('cp', 'mip'):
        try:
            ob = OptimalBinning(name=feature_name, dtype='numerical',
                                solver=solver, max_n_bins=10)
            ob.fit(X_arr, y_arr)
            iv = ob.iv
            if not (iv != iv):  # nicht NaN
                return iv
        except Exception:
            pass
    # Beide Solver gescheitert
    _iv_nan_log.append(feature_name)
    return float('nan')

def compute_mi(feat_series, y_series, is_discrete=True):
    X_arr = feat_series.values.reshape(-1, 1)
    mi = mutual_info_classif(
        X_arr, y_series.values,
        discrete_features=[is_discrete],
        random_state=SEED,
    )[0]
    return round(mi, 4)

# NB04-Referenzwerte (aus outputs/04_diagnostics/feature_diagnostics.csv)
NB04_IV = {
    'GenHlth': 0.7905, 'HighBP': 0.6095, 'BMI': 0.4694, 'Age': 0.3970,
    'HighChol': 0.3389, 'DiffWalk': 0.3170, 'Income': 0.2281, 'PhysHlth': 0.2194,
    'HeartDiseaseorAttack': 0.1939, 'Education': 0.1226, 'PhysActivity': 0.1041,
    'MentHlth': 0.0402, 'HvyAlcoholConsump': 0.0379, 'Smoker': 0.0312,
    'Veggies': 0.0234, 'Fruits': 0.0154, 'Stroke': 0.0, 'CholCheck': 0.0,
}
NB04_MI = {
    'GenHlth': 0.0441, 'HighBP': 0.0351, 'BMI': 0.0285, 'Age': 0.0203,
    'HighChol': 0.0200, 'DiffWalk': 0.0202, 'Income': 0.0136, 'PhysHlth': 0.0141,
    'HeartDiseaseorAttack': 0.0126, 'Education': 0.0075, 'PhysActivity': 0.0064,
    'MentHlth': 0.0026, 'HvyAlcoholConsump': 0.0020, 'Smoker': 0.0019,
    'Veggies': 0.0014, 'Fruits': 0.0009, 'Stroke': 0.0045, 'CholCheck': 0.0031,
}
print('Referenzwerte geladen.')

## 3. Transformationen auf X_train anwenden

### 3.1 Features für Modelldrop vormerken

NB04 dokumentiert für die folgenden drei Features Unterschreitung aller vier
Selektionsschwellen (IV, MI, KS, Spearman) oder reinen Selektionsbias:

| Feature | IV | MI | in_shortlist | Begründung |
|---|---|---|---|---|
| `AnyHealthcare` | 0.000 | 0.0001 | False | Nahezu konstant (>95 % positiv), kein Informationsgehalt |
| `NoDocbcCost` | 0.008 | 0.0005 | False | Alle Metriken unter Schwelle |
| `Sex` | 0.008 | 0.0005 | False | Alle Metriken unter Schwelle |

**Wichtig:** `AnyHealthcare` und `NoDocbcCost` werden für den `healthcare_access_index`
(Abschnitt 4.9) benötigt und erst nach dessen Konstruktion entfernt. `Sex` wird
ebenfalls als Modell-Feature entfernt, bleibt aber als Subgruppen-Variable erhalten
(Subgruppen-Analyse in NB09 lädt `Sex` direkt aus den Original-Parquet-Dateien).

`CholCheck` bleibt erhalten — es verbleibt in der Shortlist und wird für
`healthcare_access_index` benötigt.

In [ ]:
# MODEL_DROP_COLS werden erst nach Section 4.9 (healthcare_access_index) angewendet,
# da AnyHealthcare und NoDocbcCost für den Index benötigt werden.
MODEL_DROP_COLS = ['AnyHealthcare', 'NoDocbcCost', 'Sex']

print(f'Vorgemerkte Drop-Spalten (werden nach Section 4.9 entfernt): {MODEL_DROP_COLS}')
print(f'X_train Shape vor Drop: {X_train.shape}')
assert all(c in X_train.columns for c in MODEL_DROP_COLS), 'Fehlende Drop-Spalte!'
print('Sanity-Check bestanden.')

### Interpretation

`AnyHealthcare`, `NoDocbcCost` und `Sex` sind vorgemerkt, werden aber erst nach
Konstruktion von `healthcare_access_index` (Abschnitt 4.9) aus `X_train` entfernt.
`CholCheck` bleibt erhalten.

### 3.2 BMI-Capping → `BMI_capped`

**Formel:** `BMI_capped = clip(BMI, 18, 50)`

Capping auf [18, 50] orientiert sich an klinischen WHO-Schwellen (BMI < 18,5 = Untergewicht;
BMI ≥ 50 = Super-Adipositas) und den P1/P99-Perzentilen aus NB02.
Der Originalwert `BMI` bleibt erhalten, da er als Basis für nachfolgende Transformationen
(BMI_squared, BMI_yj in Abschnitt 6) benötigt wird.

BMI, BMI_capped und BMI_cat bleiben parallel erhalten — unterschiedliche Modellfamilien
nutzen unterschiedliche Granularität. Finale Auswahl in NB07.

**Quelle:** WHO Expert Consultation 2004, *Lancet*, 363(9403):157–163.

In [ ]:
X_train = cap_bmi(X_train)

n_affected_train = ((X_train['BMI'] < 18) | (X_train['BMI'] > 50)).sum()
print(f'BMI_capped Bereich: [{X_train["BMI_capped"].min():.1f}, {X_train["BMI_capped"].max():.1f}]')
print(f'Gecappte Samples   : {n_affected_train} ({n_affected_train/len(X_train):.2%})')
print(f'NaN in BMI_capped  : {X_train["BMI_capped"].isna().sum()}')
assert X_train['BMI_capped'].between(18, 50).all(), 'BMI_capped außerhalb [18, 50]!'
print('Sanity-Check bestanden.')

### Interpretation

1,38 % der Samples (2.805 Beobachtungen) hatten BMI-Werte außerhalb [18, 50] und wurden
gecappt. Der Bereich nach Capping ist [18.0, 50.0], keine fehlenden Werte entstanden.
Der Großteil der Verteilung bleibt unverändert — Capping betrifft nur die extremen Enden.

### 3.3 BMI-Kategorisierung → `BMI_cat`

**Formel:** Sieben WHO-Klassen nach Binning auf `BMI` (Original, nicht gecappt).

| Klasse | Label | BMI-Bereich |
|---|---|---|
| 0 | Untergewicht | < 18.5 |
| 1 | Normalgewicht | 18.5 – 25 |
| 2 | Übergewicht | 25 – 30 |
| 3 | Adipositas I | 30 – 35 |
| 4 | Adipositas II | 35 – 40 |
| 5 | Adipositas III | 40 – 50 |
| 6 | Super-Adipositas | > 50 |

Das ordinale `BMI_cat` ist für baumbasierte Modelle geeignet ohne Skalierung
und wird in Abschnitt 4 (Composite Features) verwendet.

**Quelle:** WHO Expert Consultation 2004, *Lancet*, 363(9403):157–163.

In [ ]:
X_train = categorize_bmi(X_train)

print('BMI_cat Häufigkeiten (X_train):')
cat_labels = {0:'Untergewicht', 1:'Normalgewicht', 2:'Übergewicht',
              3:'Adipositas I', 4:'Adipositas II', 5:'Adipositas III', 6:'Super'}
counts = X_train['BMI_cat'].value_counts().sort_index()
for k, v in counts.items():
    print(f'  {k} ({cat_labels[k]:>14}) : {v:>7,} ({v/len(X_train):.1%})')
print(f'\nNaN in BMI_cat: {X_train["BMI_cat"].isna().sum()}')
assert set(X_train['BMI_cat'].unique()).issubset({0,1,2,3,4,5,6}), 'Unerwartete Kategorie!'
print('Sanity-Check bestanden.')

### Interpretation

Die Verteilung spiegelt die BRFSS-Bevölkerung: 34,0 % Normalgewicht, 35,9 % Übergewicht,
17,5 % Adipositas I. Zusammen sind 69 % der Befragten übergewichtig oder adipös —
konsistent mit US-Bevölkerungsdaten. Die Kategorien 0 (Untergewicht, 1,2 %) und 6
(Super-Adipositas, 0,9 %) sind selten, bleiben aber erhalten.

### 3.4 Hurdle-Encoding → `MentHlth_any`, `MentHlth_days`, `PhysHlth_any`, `PhysHlth_days`

**Formel:** Für jede Count-Variable `col`:
- `col_any = (col > 0).astype(int)` — Binärindikator
- `col_days = col` — Originalanzahl (umbenannt)
- Original `col` wird entfernt.

Begründung (NB02): Beide Variablen sind stark zero-inflated (MentHlth: 69 %, PhysHlth: 63 %
Nullen). Ein einzelner Zahlenwert vermischt zwei inhaltlich verschiedene Informationen.
Das Hurdle-Encoding trennt diese explizit und erlaubt dem Modell, beide Anteile
unabhängig zu gewichten.

**Quelle:** Mullahy J. 1986, *Journal of Econometrics*, 33(3):341–365.

In [ ]:
X_train = hurdle_encode(X_train)

hurdle_cols = ['MentHlth_any', 'MentHlth_days', 'PhysHlth_any', 'PhysHlth_days']
print('Neue Spalten nach hurdle_encode:')
print(X_train[hurdle_cols].describe().T[['min','max','mean']].round(3))

assert (X_train['MentHlth_any'] == (X_train['MentHlth_days'] > 0)).all()
assert (X_train['PhysHlth_any'] == (X_train['PhysHlth_days'] > 0)).all()
assert 'MentHlth' not in X_train.columns and 'PhysHlth' not in X_train.columns
print(f'\nSpalten gesamt: {X_train.shape[1]}')
print('Sanity-Check bestanden.')

### Interpretation

30,8 % der Befragten hatten mindestens einen psychisch eingeschränkten Tag
(`MentHlth_any = 1`), 37,0 % mindestens einen physisch eingeschränkten Tag
(`PhysHlth_any = 1`). Die mittlere Anzahl betroffener Tage beträgt 3,2 (mental) bzw.
4,3 (physisch) — beide stark zero-inflated, was das Hurdle-Encoding rechtfertigt.

In [ ]:
print(f'X_train Shape nach allen Schritt-3-Transformationen: {X_train.shape}')
print(f'(Hinweis: MODEL_DROP_COLS werden erst nach Section 4.9 entfernt)')
print(f'Fehlende Werte (X_train): {X_train.isna().sum().sum()}')
print(f'Aktuelle Spalten:')
for c in X_train.columns:
    print(f'  {c}')

## 4. Composite Features

Composite Features fassen mehrere klinisch verwandte Einzel-Features zu einer
Summenvariablen zusammen. Jedes Feature ist durch eine publizierte Risiko-Skala
oder einen epidemiologischen Rahmen begründet.

Alle Composites werden auf Kopien der DataFrames gebildet und danach an
`X_train` bzw. `X_test` angehängt. Die Einzel-Features bleiben erhalten —
welche Darstellung das Modell bevorzugt, entscheidet die Feature-Selektion in NB07.

### 4.1 `cardio_comorbidity`

**Formel:**
```
cardio_comorbidity = HighBP + HighChol + HeartDiseaseorAttack + Stroke + DiffWalk
```
Wertebereich: 0 – 5 (ganzzählig).

Zählt kardiovaskuläre und mobilitätsbezogene Komorbiditäten. Jede Komponente
ist ein etablierter Risikofaktor im Framingham-Kontext.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** D'Agostino et al. 2008, *Circulation*, 117(6):743–753 (Framingham Risk Score).

In [ ]:
X_train['cardio_comorbidity'] = (
    X_train['HighBP'] + X_train['HighChol'] + X_train['HeartDiseaseorAttack']
    + X_train['Stroke'] + X_train['DiffWalk']
)

print('cardio_comorbidity (X_train):')
print(X_train['cardio_comorbidity'].value_counts().sort_index().to_string())
print(f'NaN: {X_train["cardio_comorbidity"].isna().sum()}')
assert X_train['cardio_comorbidity'].between(0, 5).all()
print('Sanity-Check bestanden.')

### Interpretation

Die Mehrheit (72.896 Samples, 35,9 %) hat keine der fünf Komorbiditäten. Mit
steigender Anzahl nimmt die Häufigkeit ab. Nur 1.266 Samples (0,6 %) weisen alle
fünf Faktoren gleichzeitig auf. Der Score erlaubt eine feingranulare Abstufung des
kardiovaskulären Komorbiditätsprofils.

### 4.2 `allostatic_load`

**Formel:**
```
allostatic_load = HighBP + HighChol + BMI_cat + PhysHlth_any + MentHlth_any
```
Wertebereich: 0 – 10 (abhängig von BMI_cat-Maximum 6).

Allostatic Load bezeichnet die kumulative physiologische Belastung durch chronischen
Stress. Die Kombination aus kardiometabolischen Markern (HighBP, HighChol), körperlicher
Konstitution (BMI_cat) und subjektivem Gesundheitserleben (PhysHlth_any, MentHlth_any)
folgt dem Operationalisierungsrahmen von McEwen.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** McEwen & Stellar 1993, *Archives of Internal Medicine*, 153(18):2093–2101.

In [ ]:
X_train['allostatic_load'] = (
    X_train['HighBP'] + X_train['HighChol'] + X_train['BMI_cat']
    + X_train['PhysHlth_any'] + X_train['MentHlth_any']
)

print('allostatic_load Verteilung (X_train):')
print(X_train['allostatic_load'].describe().round(2))
print(f'NaN: {X_train["allostatic_load"].isna().sum()}')
print('Sanity-Check bestanden.')

### Interpretation

Mittlerer Allostatic Load: 3,62 (SD 1,79), Bereich 0–10. Das BMI_cat dominiert den
Score numerisch (0–6), während die vier binären Komponenten maximal 4 Punkte beitragen.
Die Verteilung ist rechtschief — der Median liegt bei 3, was bedeutet, dass die Hälfte
der Befragten einen moderaten bis niedrigen Load hat.

### 4.3 `healthy_lifestyle`

**Formel:**
```
healthy_lifestyle = PhysActivity + Fruits + Veggies + (1 - Smoker) + (1 - HvyAlcoholConsump)
```
Wertebereich: 0 – 5.

Anlehnung an AHA's *Life's Simple 7*: physische Aktivität, Ernährung
(Obst + Gemüse) und Rauchverzicht gelten als zentrale verhaltensbasierte
Schutzfaktoren. Höhere Werte bedeuten gesündere Lebensweise.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** Lloyd-Jones et al. 2010, *Circulation*, 121(4):586–613.

In [ ]:
X_train['healthy_lifestyle'] = (
    X_train['PhysActivity'] + X_train['Fruits'] + X_train['Veggies']
    + (1 - X_train['Smoker']) + (1 - X_train['HvyAlcoholConsump'])
)

print('healthy_lifestyle Verteilung (X_train):')
print(X_train['healthy_lifestyle'].value_counts().sort_index().to_string())
assert X_train['healthy_lifestyle'].between(0, 5).all()
print('Sanity-Check bestanden.')

### Interpretation

Der Modus liegt bei 4 (70.300 Samples, 34,6 %): Die meisten Befragten erfüllen vier
der fünf Lifestyle-Kriterien. Nur 380 Samples (0,2 %) haben einen Score von 0 (kein
Kriterium erfüllt). Die Verteilung ist linksschief — das BRFSS-Sample ist in dieser
Hinsicht relativ gesund.

### 4.4 `metsyn_proxy`

**Formel:**
```
metsyn_proxy = HighBP + HighChol + (BMI_capped >= 30).astype(int) + HeartDiseaseorAttack
```
Wertebereich: 0 – 4.

Approximiert das Metabolische Syndrom mit BRFSS-verfügbaren Variablen.
BMI ≥ 30 entspricht dem IDF-Kriterium für zentrales Übergewicht als Kerndefinition.
Glukose, Triglyceride und HDL liegen im BRFSS nicht vor.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** International Diabetes Federation 2005,
*The IDF consensus worldwide definition of the metabolic syndrome*.

In [ ]:
X_train['metsyn_proxy'] = (
    X_train['HighBP'] + X_train['HighChol']
    + (X_train['BMI_capped'] >= 30).astype(int)
    + X_train['HeartDiseaseorAttack']
)

print('metsyn_proxy Verteilung (X_train):')
print(X_train['metsyn_proxy'].value_counts().sort_index().to_string())
assert X_train['metsyn_proxy'].between(0, 4).all()
print('Sanity-Check bestanden.')

### Interpretation

Die Verteilung ist annähernd gleichmäßig mit einem Peak bei 1 Kriterium (61.073 Samples).
28,2 % haben BMI ≥ 30, HighBP und HighChol gleichzeitig (Score ≥ 3) — das ist die
Hochrisiko-Gruppe im MetSyn-Kontext. Score 0 (29,5 %) entspricht Personen ohne eines
der vier messbaren Kriterien.

### 4.5 `ses_index`

**Formel:**
```
ses_index = Education + Income
```
Wertebereich: 2 – 14 (Education: 1–6, Income: 1–8).

Sozialer Status ist ein starker Diabetesprädiktor über multiple Mechanismen
(Ernährungszugang, Stresslevel, Gesundheitskompetenz). Education und Income
sind die einzigen SES-Proxies im BRFSS-Datensatz und weisen laut NB04 eine
moderate Korrelation auf (Cluster: SES).

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** Adler et al. 1994, *JAMA*, 269(24):3140–3145.

In [ ]:
X_train['ses_index'] = X_train['Education'] + X_train['Income']

print('ses_index Verteilung (X_train):')
print(X_train['ses_index'].describe().round(2))
print(f'Bereich: {X_train["ses_index"].min()} – {X_train["ses_index"].max()}')
print('Sanity-Check bestanden.')

### Interpretation

Mittlerer SES-Index: 11,10 (SD 2,67), Median 12. Der Bereich liegt erwartungsgemäß bei
2–14. Das BRFSS-Sample ist leicht zu den höheren SES-Stufen verschoben (Telefonsurvey-Bias:
schlechter erreichbare Bevölkerungsgruppen sind unterrepräsentiert).

### 4.6 `mental_physical_burden`

**Formel:**
```
mental_physical_burden = MentHlth_days + PhysHlth_days
```
Wertebereich: 0 – 60.

Addiert die Anzahl der mental und physisch eingeschränkten Tage zum Gesamtmaß
der subjektiven Gesundheitsbeeinträchtigung. Entspricht dem Summary-Score-Ansatz
des SF-36-Instruments zur gesundheitsbezogenen Lebensqualität.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** Ware & Sherbourne 1992, *Medical Care*, 30(6):473–483 (SF-36).

In [ ]:
X_train['mental_physical_burden'] = X_train['MentHlth_days'] + X_train['PhysHlth_days']

print('mental_physical_burden Verteilung (X_train):')
print(X_train['mental_physical_burden'].describe().round(2))
assert X_train['mental_physical_burden'].between(0, 60).all()
print('Sanity-Check bestanden.')

### Interpretation

Mittlerer Gesamtburden: 7,44 Tage (SD 13,30). Der Median liegt bei 1 — die Verteilung
ist stark zero-inflated (viele Nullen). Der Maximalwert 60 entspricht Personen, die in
beiden Dimensionen je 30 Tage beeinträchtigt waren. Der Score ist stark linksschief;
für lineare Modelle bietet sich eine log-Transformation an (siehe Abschnitt 6.3).

### 4.7 `findrisc_lite`

**Formel:**
```
findrisc_lite = BMI_cat + Age + (1 - PhysActivity) + (1 - Fruits) + (1 - Veggies)
```
Wertebereich: 0 – 14 (BMI_cat 0–6, Age 1–13).

Vereinfachte Adaptation des Finnish Diabetes Risk Score (FINDRISC).
Höhere Werte bedeuten höheres Diabetesrisiko. FINDRISC ist ein validierter
Fragebogen-basierter Screening-Score für Typ-2-Diabetes.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** Lindström & Tuomilehto 2003, *Diabetes Care*, 26(3):725–731.

In [ ]:
X_train['findrisc_lite'] = (
    X_train['BMI_cat'] + X_train['Age']
    + (1 - X_train['PhysActivity']) + (1 - X_train['Fruits']) + (1 - X_train['Veggies'])
)

print('findrisc_lite Verteilung (X_train):')
print(X_train['findrisc_lite'].describe().round(2))
print(f'Bereich: {X_train["findrisc_lite"].min()} – {X_train["findrisc_lite"].max()}')
print('Sanity-Check bestanden.')

### Interpretation

Mittlerer FINDRISC-lite-Score: 10,92 (SD 3,39), Bereich 1–22. Age dominiert den Score
numerisch (Wertebereich 1–13). Der Score streut breit, was auf gute Differenzierungskraft
hindeutet. Der FINDRISC misst Alters- und Körpergewichtslast kombiniert mit
Ernährungsverhalten — Variablen, die laut NB04 zu den stärksten Prädiktoren zählen.

### 4.8 `ascvd_proxy`

**Formel:**
```
ascvd_proxy = Age + HighBP + HighChol + Smoker + HeartDiseaseorAttack
```
Wertebereich: 1 – 17 (Age 1–13).

Approximiert das arteriosklerotische Kardiovaskuläre Erkrankungsrisiko
mit den im BRFSS verfügbaren Risikofaktoren der ACC/AHA Pooled Cohort Equations.
Alter ist der dominante Koeffizient, ergänzt durch binäre Risikofaktoren.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** Goff et al. 2014, *JACC*, 63(25 Pt B):2935–2959 (ACC/AHA PCE).

In [ ]:
X_train['ascvd_proxy'] = (
    X_train['Age'] + X_train['HighBP'] + X_train['HighChol']
    + X_train['Smoker'] + X_train['HeartDiseaseorAttack']
)

print('ascvd_proxy Verteilung (X_train):')
print(X_train['ascvd_proxy'].describe().round(2))
print(f'Bereich: {X_train["ascvd_proxy"].min()} – {X_train["ascvd_proxy"].max()}')
print('Sanity-Check bestanden.')

### Interpretation

Mittlerer ASCVD-Proxy: 9,42 (SD 3,62), Bereich 1–17. Age dominiert auch hier numerisch.
Der Score korreliert stark mit findrisc_lite (beide enthalten Age und Lifestyle-Faktoren),
misst aber den kardiovaskulären Pfad expliziter. Ob er nach Kontrolle für findrisc_lite
eigenständige Varianz erklärt, entscheidet die Feature-Selektion in NB07.

### 4.9 `healthcare_access_index`

**Formel:**
```
healthcare_access_index = CholCheck + AnyHealthcare + (1 - NoDocbcCost)
```
Wertebereich: 0 – 3 (ganzzählig).

Proxy für den Zugang zum Gesundheitssystem und die Wahrscheinlichkeit einer Diagnose.
Höhere Werte bedeuten wahrscheinlicheren Systemkontakt: Cholesterin-Check (Arztbesuch),
Krankenversicherung (AnyHealthcare) und keine finanziellen Hürden (NoDocbcCost = 0).
Kein biologischer Risikofaktor — misst Diagnose-Wahrscheinlichkeit, nicht Krankheitsrisiko.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quellen:**
- Andersen & Newman 1973, *Milbank Memorial Fund Quarterly*, 51(1):95–124
- Levesque et al. 2013, *BMC Health Services Research*, 13:276

In [ ]:
X_train['healthcare_access_index'] = (
    X_train['CholCheck'] + X_train['AnyHealthcare'] + (1 - X_train['NoDocbcCost'])
)

print('healthcare_access_index Verteilung (X_train):')
print(X_train['healthcare_access_index'].value_counts().sort_index().to_string())
print(f'Bereich: {X_train["healthcare_access_index"].min()} – {X_train["healthcare_access_index"].max()}')
print(f'NaN: {X_train["healthcare_access_index"].isna().sum()}')
assert X_train['healthcare_access_index'].between(0, 3).all(), 'Bereich außerhalb [0, 3]!'
assert X_train['healthcare_access_index'].isna().sum() == 0, 'NaN gefunden!'
print('Sanity-Check bestanden.')

### 4.10 `ses_x_access`

**Formel:**
```
ses_x_access = ses_index * healthcare_access_index
```
Interaktion zwischen sozioökonomischem Status und Gesundheitssystemzugang.

Diabetesdiagnose-Wahrscheinlichkeit hängt nicht nur von Krankheitsrisiko ab, sondern
von sozialer Lage (SES) und Systemzugang (healthcare_access_index) gemeinsam. Personen
mit niedrigem SES und schlechtem Zugang haben besonders hohe Unterdiagnose-Wahrscheinlichkeit.

Diese Variable ist eine BRFSS-basierte Proxy-Approximation — kein validierter klinischer Score.

**Quelle:** Phelan et al. 2010, *Journal of Health and Social Behavior*, 51(Suppl):S28–S40.

In [ ]:
X_train['ses_x_access'] = X_train['ses_index'] * X_train['healthcare_access_index']

print('ses_x_access Verteilung (X_train):')
print(X_train['ses_x_access'].describe().round(2))
print(f'Bereich: {X_train["ses_x_access"].min()} – {X_train["ses_x_access"].max()}')
print(f'NaN: {X_train["ses_x_access"].isna().sum()}')
assert X_train['ses_x_access'].isna().sum() == 0, 'NaN gefunden!'
print('Sanity-Check bestanden.')

In [ ]:
# MODEL_DROP_COLS jetzt anwenden: AnyHealthcare und NoDocbcCost wurden für
# healthcare_access_index (4.9) benötigt und können jetzt entfernt werden.
X_train = X_train.drop(columns=MODEL_DROP_COLS)

print(f'Spalten nach Model-Drop: {X_train.shape[1]}  (entfernt: {MODEL_DROP_COLS})')
assert not any(c in X_train.columns for c in MODEL_DROP_COLS), 'Drop unvollständig!'
print('Sanity-Check bestanden.')

COMPOSITE_FEATURES = [
    'cardio_comorbidity', 'allostatic_load', 'healthy_lifestyle', 'metsyn_proxy',
    'ses_index', 'mental_physical_burden', 'findrisc_lite', 'ascvd_proxy',
    'healthcare_access_index', 'ses_x_access',
]
print(f'\nShape X_train nach Section 4: {X_train.shape}')
print(f'Composite Features: {len(COMPOSITE_FEATURES)}')

## 5. Interaktionsterme

Interaktionsterme modellieren nicht-additive Effekte zweier Features.
Sie werden als einfache Produkte berechnet — Baummodelle können solche
Interaktionen grundsätzlich selbst lernen, aber explizite Terme beschleunigen
die Konvergenz und erhöhen die Interpretierbarkeit linearer Ableitungen.

Alle vier Terme sind epidemiologisch motiviert.

### 5.1 `BMI_x_Age`

**Formel:** `BMI_x_Age = BMI_capped × Age`

Der Effekt von Übergewicht auf Diabetes ist altersabhängig: Bei jüngeren Personen
ist der BMI-Effekt stärker ausgeprägt als bei älteren, wo Sarkopenie den BMI-Wert
verzerren kann.

**Quelle:** Janssen et al. 2005, *Obesity Research*, 13(12):2072–2079.

In [ ]:
X_train['BMI_x_Age'] = X_train['BMI_capped'] * X_train['Age']

print('BMI_x_Age (X_train):')
print(X_train['BMI_x_Age'].describe().round(2))

### Interpretation

Mittlerer BMI×Age: 226,59 (SD 95,97), Bereich 18–650. Die hohe Streuung spiegelt die
kombinierte Varianz beider Komponenten wider. Für lineare Modelle sollte das Feature
standardisiert werden; Baummodelle können es direkt verwenden.

### 5.2 `BMI_x_HighBP`

**Formel:** `BMI_x_HighBP = BMI_capped × HighBP`

Hypertonie und Adipositas potenzieren sich gegenseitig im Diabetesrisiko;
der kombinierte Effekt übertrifft die Summe der Einzeleffekte.

**Quelle:** Landsberg et al. 2013, *Journal of Clinical Hypertension*, 15(1):14–33.

In [ ]:
X_train['BMI_x_HighBP'] = X_train['BMI_capped'] * X_train['HighBP']

print('BMI_x_HighBP (X_train):')
print(X_train['BMI_x_HighBP'].describe().round(2))

### Interpretation

Für Personen ohne HighBP (HighBP = 0) ist das Feature exakt 0 — 50 % der Samples.
Für die andere Hälfte entspricht der Wert BMI_capped (Bereich 18–50). Der Median von 0
zeigt, dass nur für den hypertensiven Teil der Bevölkerung ein Interaktionssignal vorliegt.

### 5.3 `Age_x_GenHlth`

**Formel:** `Age_x_GenHlth = Age × GenHlth`

GenHlth (allgemeiner Gesundheitszustand) und Alter sind die beiden stärksten
Einzelprädiktoren (NB04: IV = 0.79 bzw. 0.40). Ihr Produkt erfasst, dass schlechter
Gesundheitszustand im Alter besonders stark mit Diabetes assoziiert ist.

**Quelle:** Hu et al. 2001, BRFSS Diabetes Predictors Study.

In [ ]:
X_train['Age_x_GenHlth'] = X_train['Age'] * X_train['GenHlth']

print('Age_x_GenHlth (X_train):')
print(X_train['Age_x_GenHlth'].describe().round(2))

### Interpretation

Mittlerer Age×GenHlth: 20,67 (SD 12,74), Bereich 1–65. Das Feature kombiniert die
zwei stärksten Prädiktoren aus NB04 (IV 0,79 und 0,40). Die Streuung ist breit, was
gute Diskriminationskraft verspricht. Für lineare Modelle ist Standardisierung empfohlen.

### 5.4 `HighBP_x_HighChol`

**Formel:** `HighBP_x_HighChol = HighBP × HighChol`

Das gleichzeitige Vorliegen von Hypertonie und Hypercholesterinämie ist ein
Kernkriterium des Metabolischen Syndroms. Das Produkt ist 1 nur wenn beide
Bedingungen zutreffen (logisches AND bei Binärvariablen).

**Quelle:** International Diabetes Federation 2005,
*The IDF consensus worldwide definition of the metabolic syndrome*.

In [ ]:
X_train['HighBP_x_HighChol'] = X_train['HighBP'] * X_train['HighChol']

print('HighBP_x_HighChol (X_train):')
print(X_train['HighBP_x_HighChol'].value_counts().sort_index().to_string())

INTERACTION_FEATURES = ['BMI_x_Age', 'BMI_x_HighBP', 'Age_x_GenHlth', 'HighBP_x_HighChol']
print(f'\nShape X_train nach Section 5: {X_train.shape}')

### Interpretation

151.143 Samples (74,5 %) haben 0 — mindestens eine der beiden Bedingungen fehlt.
51.801 (25,5 %) haben beide Erkrankungen gleichzeitig. Das Feature entspricht einem
logischen AND und ist als binäres Feature nützlich für Entscheidungsbäume.

## 6. Polynomterme und nichtlineare Transformationen

BMI zeigt in epidemiologischen Studien einen nichtlinearen Zusammenhang mit
dem Diabetesrisiko (J- bzw. U-Kurve im unteren Bereich, steile Kurve im oberen).
Für die Count-Variablen (MentHlth_days, PhysHlth_days) nivelliert die log-Transformation
die zero-inflated Rechtsschiefe für lineare Modelle.

### 6.1 `BMI_squared`

**Formel:** `BMI_squared = BMI_capped²`

Das Quadrat von BMI ermöglicht linearen Modellen, den überproportionalen Anstieg
des Diabetesrisikos bei hohen BMI-Werten zu modellieren.

**Quelle:** Tirosh et al. 2011, *NEJM*, 364(15):1315–1325.

In [ ]:
X_train['BMI_squared'] = X_train['BMI_capped'] ** 2

print('BMI_squared (X_train):')
print(X_train['BMI_squared'].describe().round(2))

### Interpretation

BMI_squared Bereich: 324–2500 (entspricht BMI 18–50 nach Capping). Mittlerer Wert 836
(≈ BMI 29²). Das Feature verstärkt den Effekt hoher BMI-Werte überproportional —
nützlich für lineare Modelle, die den nonlinearen Anstieg des Diabetesrisikos bei
Adipositas abbilden müssen.

### 6.2 `BMI_yj` — Yeo-Johnson-Transformation

**Formel:** `BMI_yj = PowerTransformer(method='yeo-johnson').fit_transform(BMI_capped)`

Die Yeo-Johnson-Transformation (Verallgemeinerung von Box-Cox auf negative Werte)
normalisiert die BMI-Verteilung. Der Transformer wird **ausschließlich auf X_train**
gefittet und dann auf X_test angewendet — kein Datenleck.

Das gefittete `PowerTransformer`-Objekt wird unter `models/transformers/power_transformer.pkl`
gespeichert und in NB07 (Pipelines) wiederverwendet.

**Quelle:** Yeo & Johnson 2000, *Biometrika*, 87(4):954–959.

In [ ]:
pt = PowerTransformer(method='yeo-johnson', standardize=True)

# Fit nur auf X_train! Anwendung auf X_test erfolgt in NB07-Pipelines.
X_train['BMI_yj'] = pt.fit_transform(X_train[['BMI_capped']]).ravel()

print('PowerTransformer Lambda:', pt.lambdas_)
print('BMI_yj (X_train):')
print(X_train['BMI_yj'].describe().round(4))
print(f'NaN in BMI_yj: {X_train["BMI_yj"].isna().sum()}')

### Interpretation

Lambda ≈ −0,83 deutet auf eine inverse Box-Cox-ähnliche Transformation hin — die
rechtsschiefe BMI-Verteilung wird annähernd normalisiert (mean ≈ 0, std ≈ 1).
Kein NaN. Der gefittete Transformer wird in NB07 auf X_test angewendet.

### 6.3 `MentHlth_log` und `PhysHlth_log`

**Formel:** `log1p(MentHlth_days)`, `log1p(PhysHlth_days)`

`log1p(x) = ln(1 + x)` ist für `x = 0` definiert (→ 0) und behandelt damit die
vielen Nullen in den Count-Variablen korrekt. Empfohlen für zero-inflated Zähldaten
in linearen Modellen.

**Quelle:** Mullahy J. 1986, *Journal of Econometrics*, 33(3):341–365.

In [ ]:
X_train['MentHlth_log'] = np.log1p(X_train['MentHlth_days'])
X_train['PhysHlth_log'] = np.log1p(X_train['PhysHlth_days'])

print('MentHlth_log (X_train):')
print(X_train['MentHlth_log'].describe().round(4))
print('\nPhysHlth_log (X_train):')
print(X_train['PhysHlth_log'].describe().round(4))

NONLINEAR_FEATURES = ['BMI_squared', 'BMI_yj', 'MentHlth_log', 'PhysHlth_log']
print(f'\nShape X_train nach Section 6: {X_train.shape}')

### Interpretation

Beide log-transformierten Variablen behalten ihre Nullen korrekt (log1p(0) = 0). Die
mittlere MentHlth_log liegt bei 0,62, PhysHlth_log bei 0,77 — eine deutliche
Komprimierung der Rechtsschiefe. Für lineare Modelle verbessert das die Linearitäts-
annahme zwischen diesen Features und dem Target erheblich.

## 7. IV/MI Vorher-Nachher-Vergleich

Für jedes neue Feature werden IV (OptimalBinning) und MI
(mutual_info_classif) berechnet und mit dem jeweils stärksten Einzel-Feature
unter den Komponenten verglichen (Werte aus NB04).

**Dieser Vergleich ist ein Vorfilter — keine finale Feature-Wahrheit.**

**Entscheidungsregel:** Ein neues Feature wird beibehalten, wenn
`IV_neu > IV_alt ODER MI_neu > MI_alt`.

**Wichtiger Hinweis zu IV = NaN:** OptimalBinning kann für bestimmte
Feature-Typen (nicht-monotone, niedrige Varianz, ganzzahlige Scores) keinen
stabilen Binning-Plan berechnen und gibt NaN zurück. IV = NaN ist **kein**
Selektionskriterium — in diesen Fällen entscheidet ausschließlich MI.

**Finale Feature-Auswahl erfolgt in der Modellierungsphase über
Cross-Validation / PR-AUC sowie BorutaSHAP/RFECV in NB07.**

In [ ]:
# Referenzwerte für hurdle-encodierte Zählfeatures live berechnen
# (MentHlth/PhysHlth existieren nach hurdle_encode nicht mehr im DataFrame)
_hurdle_refs = ['MentHlth_days', 'PhysHlth_days']
for _col in _hurdle_refs:
    NB04_IV[_col] = compute_iv(X_train[_col], y_train, _col)
    NB04_MI[_col] = compute_mi(X_train[_col], y_train, is_discrete=True)
    print(f'{_col}: IV={NB04_IV[_col]:.4f}  MI={NB04_MI[_col]:.4f}')

# Referenz: bestes Einzelfeature pro neuem Feature
REFERENCE = {
    # Composite
    'cardio_comorbidity':       ('HighBP',               False),
    'allostatic_load':          ('HighBP',               False),
    'healthy_lifestyle':        ('PhysActivity',         False),
    'metsyn_proxy':             ('HighBP',               False),
    'ses_index':                ('Income',               False),
    'mental_physical_burden':   ('PhysHlth',             False),
    'findrisc_lite':            ('BMI',                  False),
    'ascvd_proxy':              ('HighBP',               False),
    'healthcare_access_index':  ('CholCheck',            False),
    'ses_x_access':             ('ses_index',            False),
    # Interaktionen
    'BMI_x_Age':                ('BMI',                  True),
    'BMI_x_HighBP':             ('HighBP',               True),
    'Age_x_GenHlth':            ('GenHlth',              True),
    'HighBP_x_HighChol':        ('HighBP',               False),
    # Nichtlinear — Referenz sind die hurdle-encodierten _days-Features
    'BMI_squared':              ('BMI',                  True),
    'BMI_yj':                   ('BMI',                  True),
    'MentHlth_log':             ('MentHlth_days',        True),
    'PhysHlth_log':             ('PhysHlth_days',        True),
}

NEW_FEATURES = COMPOSITE_FEATURES + INTERACTION_FEATURES + NONLINEAR_FEATURES

print(f'Neue Features gesamt: {len(NEW_FEATURES)}')
print('Starte IV/MI-Berechnung ...')

In [ ]:
results = []
for feat in NEW_FEATURES:
    ref_feat, is_cont = REFERENCE[feat]
    iv_neu = compute_iv(X_train[feat], y_train, feat, is_continuous=is_cont)
    mi_neu = compute_mi(X_train[feat], y_train, is_discrete=not is_cont)
    iv_alt = NB04_IV.get(ref_feat, np.nan)
    mi_alt = NB04_MI.get(ref_feat, np.nan)
    behalten = (iv_neu > iv_alt) or (mi_neu > mi_alt)
    results.append({
        'Feature':            feat,
        'IV_neu':             round(iv_neu, 4),
        'MI_neu':             round(mi_neu, 4),
        'Bestes_Einzelfeat':  ref_feat,
        'IV_alt':             round(iv_alt, 4),
        'MI_alt':             round(mi_alt, 4),
        'behalten':           behalten,
    })
    print(f'  {feat:<28} IV={iv_neu:.4f}  MI={mi_neu:.4f}  {'✓' if behalten else '✗'}')

comparison_df = pd.DataFrame(results)
print(f'\nFertig. {comparison_df["behalten"].sum()} / {len(comparison_df)} Features beibehalten.')

# NaN-Report: nur neue Features (nicht Referenzberechnungen)
_nan_new = [f for f in _iv_nan_log if f in NEW_FEATURES]
if _nan_new:
    print()
    print(f'IV-NaN ({len(_nan_new)} Features) — Entscheidung basiert ausschließlich auf MI:')
    for _f in _nan_new:
        _row = comparison_df[comparison_df['Feature'] == _f].iloc[0]
        _decision = 'beibehalten (MI)' if _row['behalten'] else 'entfernt (MI)'
        _mi_n = _row['MI_neu']
        _mi_a = _row['MI_alt']
        print(f'  {_f:<28} MI_neu={_mi_n:.4f}  MI_alt={_mi_a:.4f}  → {_decision}')
else:
    print('IV für alle Features erfolgreich berechnet.')

In [ ]:
print(comparison_df.to_string(index=False))

In [ ]:
# Balkendiagramm IV Vorher-Nachher
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = range(len(comparison_df))
labels = comparison_df['Feature'].tolist()
colors = ['#2ecc71' if b else '#e74c3c' for b in comparison_df['behalten']]

# IV
axes[0].bar([i - 0.2 for i in x], comparison_df['IV_neu'], width=0.4,
            label='IV neu', color=colors, alpha=0.85)
axes[0].bar([i + 0.2 for i in x], comparison_df['IV_alt'], width=0.4,
            label='IV Referenz', color='#3498db', alpha=0.5)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
axes[0].set_title('IV — neue Features vs. Referenz')
axes[0].set_ylabel('Information Value')
axes[0].legend()

# MI
axes[1].bar([i - 0.2 for i in x], comparison_df['MI_neu'], width=0.4,
            label='MI neu', color=colors, alpha=0.85)
axes[1].bar([i + 0.2 for i in x], comparison_df['MI_alt'], width=0.4,
            label='MI Referenz', color='#3498db', alpha=0.5)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
axes[1].set_title('MI — neue Features vs. Referenz')
axes[1].set_ylabel('Mutual Information')
axes[1].legend()

fig.suptitle('IV/MI Vorher-Nachher-Vergleich — neue Features', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'iv_mi_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Features droppen, die weder IV noch MI-Kriterium erfüllen (nur X_train)
DROP_ENGINEERED = comparison_df.loc[~comparison_df['behalten'], 'Feature'].tolist()
KEEP_ENGINEERED = comparison_df.loc[comparison_df['behalten'],  'Feature'].tolist()

if DROP_ENGINEERED:
    print(f'Werden entfernt ({len(DROP_ENGINEERED)}): {DROP_ENGINEERED}')
    X_train = X_train.drop(columns=DROP_ENGINEERED)
else:
    print('Alle neuen Features beibehalten.')

print(f'Shape X_train nach Filterung: {X_train.shape}')
comparison_df.to_csv(OUTPUTS_DIR / 'iv_mi_comparison.csv', index=False)
print('Vergleichstabelle gespeichert.')

### Interpretation

**IV = NaN für alle neuen Features:** OptimalBinning konnte für keinen der neuen
Scores einen stabilen Binning-Plan berechnen (nicht-monotone ganzzahlige Summenscores,
gemischte Skalenniveaus). IV = NaN ist kein Selektionskriterium — die Entscheidung
basiert ausschließlich auf MI. Dies ist kein Artefakt des Modells, sondern eine
bekannte Einschränkung von OptimalBinning bei dieser Feature-Klasse.

**MI-Ergebnisse:** Die stärksten neuen Features sind Age_x_GenHlth (MI 0,060),
metsyn_proxy (0,059) und cardio_comorbidity (0,056) — alle deutlich über ihren
Referenz-Einzelfeatures. Der IV/MI-Filter ist ein liberaler Vorfilter; finale
Selektionsentscheidungen treffen BorutaSHAP/RFECV in NB07.

### 7.1 Event Rate Plot — Diabetesprävalenz pro Feature-Wert

Für jeden Composite (4.1–4.10) wird die Diabetes-Prävalenz pro Feature-Ausprägung
dargestellt: `X_train.groupby(feat)[y_train].mean()`. Ein monotoner Anstieg zeigt,
dass der Score das Diabetesrisiko gut ordnet.

In [ ]:
_kept_composites = [f for f in COMPOSITE_FEATURES if f in X_train.columns]

n_cols = 3
n_rows = (len(_kept_composites) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(_kept_composites):
    event_rate = y_train.groupby(X_train[feat]).mean()
    axes[i].bar(event_rate.index, event_rate.values, color='steelblue', edgecolor='white')
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_xlabel('Feature-Wert')
    axes[i].set_ylabel('Diabetes-Prävalenz')
    axes[i].set_ylim(0, event_rate.max() * 1.2)
    axes[i].axhline(y_train.mean(), color='red', linestyle='--', linewidth=0.8,
                    label=f'Basis ({y_train.mean():.2f})')
    axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Event Rate: Diabetesprävalenz pro Composite-Feature-Wert (X_train)',
             fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'event_rate_composites.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot gespeichert: event_rate_composites.png')

## 8. Export

Das angereicherte Trainingsset wird als Parquet-Datei gespeichert.
Das Testset wird in diesem Notebook **nicht** exportiert — es bleibt unverändert
und wird in NB07 innerhalb der Modellierungs-Pipelines auf Basis der hier gefitteten
Transformer transformiert.

**Gespeicherte Dateien:**
- `data/processed/X_train_enriched.parquet`
- `data/processed/feature_meta_enriched.json`
- `models/transformers/power_transformer.pkl`

In [ ]:
# Nur X_train_enriched exportieren — X_test wird in diesem Notebook nicht verändert
X_train.to_parquet(PROCESSED_DIR / 'X_train_enriched.parquet', index=True)

print(f'X_train_enriched : {X_train.shape}')
print(f'X_test           : {X_test.shape} (unverändert, kein Export)')

In [ ]:
# Feature-Metadaten
base_cols = [c for c in X_train.columns if c in (
    BINARY_COLS + ORDINAL_COLS + ['BMI']
) and c not in ['MentHlth', 'PhysHlth'] + MODEL_DROP_COLS]

meta_enriched = {
    'seed': SEED,
    'base_features': base_cols,
    'bmi_features':  [f for f in ['BMI', 'BMI_capped', 'BMI_cat', 'BMI_squared', 'BMI_yj'] if f in X_train.columns],
    'hurdle_features': ['MentHlth_any', 'MentHlth_days', 'MentHlth_log',
                        'PhysHlth_any',  'PhysHlth_days',  'PhysHlth_log'],
    'composite_features': [f for f in COMPOSITE_FEATURES if f in X_train.columns],
    'interaction_features': [f for f in INTERACTION_FEATURES if f in X_train.columns],
    'nonlinear_features':   [f for f in NONLINEAR_FEATURES   if f in X_train.columns],
    'all_features': list(X_train.columns),
    'model_drop_cols': MODEL_DROP_COLS,
    'subgroup_cols': ['Sex', 'Age', 'Income', 'Education'],
    'dropped_original':   MODEL_DROP_COLS,
    'dropped_engineered': DROP_ENGINEERED,
}

with open(PROCESSED_DIR / 'feature_meta_enriched.json', 'w') as f:
    json.dump(meta_enriched, f, indent=2)

print('feature_meta_enriched.json gespeichert.')
print(f'Gesamte Feature-Anzahl: {len(meta_enriched["all_features"])}')
print(f'model_drop_cols: {meta_enriched["model_drop_cols"]}')
print(f'subgroup_cols:   {meta_enriched["subgroup_cols"]}')
print('Hinweis: subgroup_cols werden in NB09 direkt aus den Original-Parquet-Dateien geladen.')
for group, cols in meta_enriched.items():
    if isinstance(cols, list) and group not in ('all_features', 'model_drop_cols', 'subgroup_cols',
                                                 'dropped_original', 'dropped_engineered'):
        print(f'  {group:<24}: {len(cols)}')

In [ ]:
# PowerTransformer speichern
joblib.dump(pt, MODELS_DIR / 'power_transformer.pkl')
print(f'PowerTransformer gespeichert: {(MODELS_DIR / "power_transformer.pkl").resolve()}')

# Verifikation: Laden und kurzer Check
pt_loaded = joblib.load(MODELS_DIR / 'power_transformer.pkl')
test_val = pt_loaded.transform([[30.0]])[0][0]
print(f'Verifikation Laden: pt.transform([[30.0]]) = {test_val:.4f}')

### Interpretation

`X_train_enriched.parquet` und `feature_meta_enriched.json` sind die primären
Artefakte dieses Notebooks. `X_test` wurde geladen, aber nicht transformiert und
nicht exportiert — alle Test-Transformationen erfolgen in NB07 innerhalb der Pipelines.
Der gespeicherte `PowerTransformer` wird in NB07 für die BMI_yj-Transformation auf
X_test eingesetzt.

## 9. Zusammenfassung

Dieses Notebook hat das Feature-Set in vier Schritten angereichert. Alle
Transformationen wurden ausschließlich auf `X_train` angewendet; `X_test` wurde
geladen, aber nicht verändert und nicht exportiert.

**Schritt 1 — Vorbereitungen:** Drei schwache Features (`AnyHealthcare`, `NoDocbcCost`,
`Sex`) wurden als `MODEL_DROP_COLS` vorgemerkt — der tatsächliche Drop erfolgte erst
nach Konstruktion von `healthcare_access_index` (4.9), da beide Access-Features für
den Index benötigt wurden. `CholCheck` bleibt erhalten.

**Schritt 2 — Basistransformationen:** BMI wurde gecappt (`BMI_capped`) und
kategorisiert (`BMI_cat`). `MentHlth` und `PhysHlth` wurden per Hurdle-Encoding in
je zwei Features aufgetrennt.

**Schritt 3 — Composite Features (4.1–4.10):** Zehn klinisch begründete Scores
wurden konstruiert — darunter neu: `healthcare_access_index` (Systemzugang-Proxy)
und `ses_x_access` (SES × Systemzugang). Alle Composites sind BRFSS-basierte
Proxy-Approximationen, keine validierten klinischen Scores.

**Schritt 4 — Interaktionen und Nichtlinearitäten:** Vier Interaktionsterme und
vier nichtlineare Transformationen wurden ergänzt.

**IV/MI-Vorfilter:** IV = NaN für alle neuen Features (OptimalBinning-Einschränkung
bei ganzzahligen Summenscores) — Selektion basierte ausschließlich auf MI.
Der Vorfilter ist liberal; finale Feature-Auswahl erfolgt über Cross-Validation /
PR-AUC sowie BorutaSHAP/RFECV in NB07.

**Gespeicherte Artefakte:**
- `data/processed/X_train_enriched.parquet`
- `data/processed/feature_meta_enriched.json` (inkl. `model_drop_cols`, `subgroup_cols`)
- `models/transformers/power_transformer.pkl`
- `outputs/05_feature_engineering/iv_mi_comparison.csv`
- `outputs/05_feature_engineering/iv_mi_comparison.png`
- `outputs/05_feature_engineering/event_rate_composites.png`

**Nächster Schritt:** NB07 — Modellierung auf `X_train_enriched` mit Stratified
K-Fold Cross-Validation (PR-AUC als Primärmetrik); Feature-Selektion über
BorutaSHAP/RFECV; Threshold-Wahl auf CV-Validierungsfolds.